## snapshot אחד בכל פעם: `git init`, `add`, `commit`

ב-Git, **commit** הוא תמונת מצב (snapshot) שלמה של כל הקבצים שאתם עוקבים אחריהם, ברגע נתון — לא רשימת שינויים (diff) כמו שאפשר לחשוב. כל commit מתויג בהודעה קצרה שמסבירה מה נעשה, ונשמר לצמיתות בהיסטוריית הפרויקט.

לפני שיוצרים commit, שינויים עוברים דרך **אזור ה-staging** (ה"מגירת טיוטה") — מקום ביניים שבו בוחרים בדיוק אילו שינויים ייכנסו ל-snapshot הבא. זה מאפשר, למשל, לבחור רק חלק מהקבצים ששיניתם לצילום מצב אחד, ולהשאיר את השאר לצילום הבא.

שלוש הפקודות הבסיסיות:

- **`git init`** — הופך תיקייה רגילה ל-repository: תיקייה שבה Git עוקב אחרי היסטוריית הקבצים. מריצים פעם אחת בתחילת הפרויקט.
- **`git add <קובץ>`** — מעביר שינויים באותו קובץ לאזור ה-staging (המגירה).
- **`git commit -m "הודעה"`** — לוקח את כל מה שנמצא כרגע במגירה, ושומר אותו כתמונת מצב קבועה בהיסטוריה, עם ההודעה שכתבתם.

```{note}
כל דוגמאות ה-Git בשבוע הזה רצות בתוך תיקייה **זמנית**, שנוצרת מחדש בכל הרצה של המחברת (`tempfile.mkdtemp()`) — כדי לא לגעת בפרויקט האמיתי שלכם. כשתתרגלו על המחשב שלכם, תריצו את אותן פקודות בדיוק בתוך תיקיית הפרויקט האמיתית.
```

### דוגמה: לעטוף repo סביב סקריפט אמיתי

ניקח את סקריפט טווח הזריקה שכבר בנינו בסעיף 7.10 (`R = v0**2 * sin(2*theta) / g`), ונבנה אותו **בהדרגה, קומיט אחרי קומיט** — בדיוק כמו שהייתם עושים תוך כדי עבודה אמיתית: קודם השלד, אחר כך החישוב, ולבסוף ההדפסה.

In [ ]:
import tempfile, os

workdir = tempfile.mkdtemp(prefix="git_demo_")
os.chdir(workdir)
print("עובדים בתיקייה זמנית:", workdir)

In [ ]:
!git init -q -b main
!git config user.email "student@example.com"
!git config user.name "Student"
!git config color.ui false
!git status

`git status` מדווח: repo ריק, בלי קבצים עוקבים. עכשיו ניצור את השלד של הסקריפט, ונבצע קומיט ראשון.

In [ ]:
%%writefile projectile.py
import numpy as np
from scipy import constants

v0 = 20.0
theta_deg = 45

In [ ]:
!git add projectile.py
!git commit -q -m "שלד: הגדרת המשתנים v0 ו-theta_deg"
!git log --oneline

עכשיו נוסיף את חישוב הטווח עצמו — **אותו קובץ**, שינוי חדש, קומיט חדש.

In [ ]:
%%writefile -a projectile.py

theta = np.radians(theta_deg)
R = v0**2 * np.sin(2 * theta) / constants.g

In [ ]:
!git add projectile.py
!git commit -q -m "הוספת חישוב טווח הזריקה R"
!git log --oneline

ולבסוף, שורת הדפסה — קומיט שלישי.

In [ ]:
%%writefile -a projectile.py

print(f"R = {R:.2f} m")

In [ ]:
!git add projectile.py
!git commit -q -m "הוספת הדפסת התוצאה"
!git log --oneline
!echo "---"
!python3 projectile.py

שלושה קומיטים, כל אחד תמונת מצב שלמה של `projectile.py` באותו רגע — לא "שינוי אחרון" אלא **גרסה מלאה**. אפשר תמיד לחזור לכל אחת מהן (נלמד איך לקרוא ולהשוות ביניהן בסעיף הבא).

### בדקו את עצמכם

In [ ]:
from jupyterquiz import display_quiz

questions = [
    {
        "question": "מה בדיוק commit ב-Git?",
        "type": "multiple_choice",
        "answers": [
            {"answer": "רשימת ההבדלים (diff) בין הקובץ הישן לחדש בלבד", "correct": False, "feedback": "לא — commit הוא תמונת מצב מלאה, לא רק רשימת ההבדלים (אם כי Git יודע לחשב diff בין שני commits, זה לא איך שהם נשמרים)."},
            {"answer": "תמונת מצב (snapshot) שלמה של הקבצים העוקבים, ברגע נתון", "correct": True, "feedback": "נכון."},
            {"answer": "גיבוי אוטומטי שקורה כל כמה דקות", "correct": False, "feedback": "לא — commit קורה רק כשמריצים git commit במפורש, לא אוטומטית."},
            {"answer": "עותק של הקובץ שנשמר בענן", "correct": False, "feedback": "commit נשמר מקומית, בתיקיית ה-.git; שום דבר לא נשלח לענן עד שמבצעים push (סעיף 12.5)."}
        ]
    },
    {
        "question": "מה תפקיד `git add` לפני `git commit`?",
        "type": "multiple_choice",
        "answers": [
            {"answer": "מוסיף קובץ חדש לגמרי לפרויקט לראשונה בלבד", "correct": False, "feedback": "git add עובד גם על קבצים שכבר קיימים ב-repo ופשוט השתנו, לא רק על קבצים חדשים."},
            {"answer": "מעביר שינויים לאזור ה-staging, כדי לבחור מה בדיוק ייכנס לקומיט הבא", "correct": True, "feedback": "נכון — זה בדיוק ה'מגירה' שדיברנו עליה."},
            {"answer": "שולח את הקובץ ל-GitHub", "correct": False, "feedback": "זה תפקידה של git push, לא git add — ראו סעיף 12.5."},
            {"answer": "מוחק את הגרסה הקודמת של הקובץ", "correct": False, "feedback": "Git לעולם לא מוחק גרסאות קודמות — הן תמיד נשארות זמינות בהיסטוריה."}
        ]
    }
]
display_quiz(questions)

### תרגול עצמי

בנו repo חדש (בתיקייה זמנית נפרדת) סביב סקריפט המהירות מזריקה חופשית מסעיף 7.1 — `v(t) = v0 - g*t`. בצעו **שלושה** קומיטים אמיתיים, בדיוק כמו למעלה:

1. קומיט ראשון — שלד עם `v0` ו-`g` בלבד.
2. קומיט שני — הוספת שורת החישוב `v = v0 - g*t`.
3. קומיט שלישי — הוספת הדפסת התוצאה.

בסיום, הריצו `git log --oneline` ווודאו ששלושת הקומיטים מופיעים.

In [ ]:
# workdir2 = tempfile.mkdtemp(prefix="git_practice_")
# os.chdir(workdir2)
# !git init -q -b main
# !git config user.email "student@example.com"
# !git config user.name "Student"
#
# כתבו כאן את שלושת הקומיטים

`````{admonition} פתרון
:class: dropdown, tip
```python
workdir2 = tempfile.mkdtemp(prefix="git_practice_")
os.chdir(workdir2)
```

```python
!git init -q -b main
!git config user.email "student@example.com"
!git config user.name "Student"
```

```python
%%writefile velocity.py
v0 = 15.0
g = 9.8
```
```python
!git add velocity.py
!git commit -q -m "שלד: הגדרת v0 ו-g"
```

```python
%%writefile -a velocity.py

t = 2.0
v = v0 - g * t
```
```python
!git add velocity.py
!git commit -q -m "הוספת חישוב המהירות v"
```

```python
%%writefile -a velocity.py

print(f"v = {v:.2f} m/s")
```
```python
!git add velocity.py
!git commit -q -m "הוספת הדפסת התוצאה"
!git log --oneline
```
`````